# Bias correction of CMIP6 and DestinE precipitation for LeakyBucket

The LeakyBucket model only uses precipitation (`pr`), so bias correction is
applied to `pr` alone. ERA5 is the observational reference.

**Method:** Quantile Delta Mapping (QDM), multiplicative (`kind='*'`).

| Source | simh | simp |
|---|---|---|
| CMIP6 | CMIP6 historical (cal. period) | CMIP6 future SSPs |
| DestinE | DestinE historical (cal. period) | DestinE SSP3-7.0 future |

**Output:** a `LumpedMakkinkForcing`-compatible YAML + NetCDF saved under
`leakybucket_bias_correction/` next to each raw forcing directory.
Step 3b can load these directly with `LumpedMakkinkForcing.load()`.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import sys
import json
import shutil
import yaml
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from rich import print
from cmethods import adjust

import ewatercycle
import ewatercycle.forcing

try:
    from scripts.forcing_destine import DestinEHistoricalForcing, DestinEFutureForcing
except ImportError:
    sys.path.insert(0, str(Path().resolve().parent))
    from scripts.forcing_destine import DestinEHistoricalForcing, DestinEFutureForcing

In [ ]:
# Parameters — overridden by papermill when running on HPC
settings_path = "settings.json"

In [ ]:
with open(settings_path, 'r') as f:
    settings = json.load(f)

display(settings)

## Load ERA5 precipitation reference

In [ ]:
era5_dir = Path(settings['path_ERA5']) / 'work' / 'diagnostic' / 'script'
era5_forcing = ewatercycle.forcing.sources['LumpedMakkinkForcing'].load(directory=era5_dir)
display(era5_forcing)

cal_start = pd.to_datetime(settings['calibration_start_date']).tz_localize(None)
cal_end   = pd.to_datetime(settings['calibration_end_date']).tz_localize(None)

da_era5_pr = xr.open_dataset(era5_forcing['pr'])['pr'].sel(
    time=slice(cal_start, cal_end)
)
print(f'ERA5 pr calibration window: {cal_start} → {cal_end}  '
      f'({len(da_era5_pr.time)} timesteps)')

## Bias correction helper

In [ ]:
def bc_pr(da_obs: xr.DataArray, da_simh: xr.DataArray, da_simp: xr.DataArray) -> xr.DataArray:
    """Apply QDM (multiplicative) to precipitation.

    Parameters
    ----------
    da_obs : xr.DataArray
        ERA5 pr over calibration period.
    da_simh : xr.DataArray
        Simulated historical pr over same calibration period.
    da_simp : xr.DataArray
        Simulated future pr to be corrected.

    Returns
    -------
    xr.DataArray
        Bias-corrected future pr.
    """
    # Normalize times to tz-naive
    obs  = da_obs.copy()
    simh = da_simh.copy()
    obs.coords['time']  = pd.to_datetime(obs.time.values).tz_localize(None)
    simh.coords['time'] = pd.to_datetime(simh.time.values).tz_localize(None)

    common = np.intersect1d(obs.time.values, simh.time.values)
    obs  = obs.sel(time=obs.time.isin(common))
    simh = simh.sel(time=simh.time.isin(common))

    corrected = adjust(
        obs=obs.squeeze(),
        simh=simh.squeeze(),
        simp=da_simp.squeeze(),
        method='quantile_delta_mapping',
        kind='*',
        n_quantiles=250,
    )

    if isinstance(corrected, xr.Dataset):
        corrected = corrected['pr']
    corrected.attrs = da_simp.attrs
    return corrected


def save_lb_forcing(da_pr_bc: xr.DataArray, out_dir: Path,
                    start_time: str, end_time: str, shape_path: Path) -> None:
    """Write bias-corrected pr + LumpedMakkinkForcing YAML to out_dir."""
    out_dir.mkdir(parents=True, exist_ok=True)

    start_str = datetime.strptime(start_time, '%Y-%m-%dT%H:%M:%SZ').strftime('%Y_%m_%d')
    end_str   = datetime.strptime(end_time,   '%Y-%m-%dT%H:%M:%SZ').strftime('%Y_%m_%d')
    pr_fname  = f'LB_BC_pr_{start_str}-{end_str}.nc'

    # Strip timezone before saving
    da_save = da_pr_bc.copy()
    da_save.coords['time'] = pd.to_datetime(da_save.time.values).tz_localize(None)
    da_save.to_netcdf(out_dir / pr_fname)

    # Copy catchment shapefile and all sidecar files
    for sidecar in shape_path.parent.glob(f'{shape_path.stem}.*'):
        shutil.copy(sidecar, out_dir / sidecar.name)

    yaml_data = {
        'start_time': start_time,
        'end_time':   end_time,
        'shape':      shape_path.name,
        'filenames':  {'pr': pr_fname},
    }
    with open(out_dir / 'ewatercycle_forcing.yaml', 'w') as f:
        yaml.dump(yaml_data, f, default_flow_style=False, sort_keys=False)

## CMIP6 — load historical and future pr

In [ ]:
def load_cmip_forcing(settings, dataset, experiment, ensemble):
    path = (
        Path(settings['path_CMIP6'])
        / dataset / experiment / ensemble
        / 'work' / 'diagnostic' / 'script'
    )
    return ewatercycle.forcing.sources['LumpedMakkinkForcing'].load(directory=path)


cmip_dataset   = settings['CMIP_info']['dataset'][0]
cmip_hist_ens  = settings['CMIP_info']['ensembles'][0]

hist_forcing = load_cmip_forcing(settings, cmip_dataset, 'historical', cmip_hist_ens)
da_cmip_hist_pr = xr.open_dataset(hist_forcing['pr'])['pr'].sel(
    time=slice(cal_start, cal_end)
)
print(f'CMIP6 historical pr: {cmip_dataset}/{cmip_hist_ens}  '
      f'({len(da_cmip_hist_pr.time)} timesteps)')

In [ ]:
# Load all CMIP6 future pr
cmip_future_pr = {}  # (dataset, experiment, ensemble) → xr.DataArray

for dataset in settings['CMIP_info']['dataset']:
    for experiment in settings['CMIP_info']['experiments'][1:]:  # skip 'historical'
        for ensemble in settings['CMIP_info']['ensembles']:
            key = (dataset, experiment, ensemble)
            try:
                fo = load_cmip_forcing(settings, dataset, experiment, ensemble)
                cmip_future_pr[key] = xr.open_dataset(fo['pr'])['pr']
                print(f'Loaded: {dataset}/{experiment}/{ensemble}')
            except FileNotFoundError:
                print(f'[yellow]Skipping (not found): {dataset}/{experiment}/{ensemble}[/yellow]')

## Apply bias correction — CMIP6

In [ ]:
shape_path = Path(settings['path_shape'])
cmip_bc_dirs = {}

for (dataset, experiment, ensemble), da_simp in cmip_future_pr.items():
    print(f'\nBias correcting: {dataset}/{experiment}/{ensemble}')
    da_bc = bc_pr(da_era5_pr, da_cmip_hist_pr, da_simp)

    out_dir = (
        Path(settings['path_CMIP6'])
        / dataset / experiment / ensemble
        / 'leakybucket_bias_correction'
    )
    save_lb_forcing(
        da_bc, out_dir,
        settings['future_start_date'],
        settings['future_end_date'],
        shape_path,
    )
    cmip_bc_dirs[(dataset, experiment, ensemble)] = out_dir
    print(f'  Saved → {out_dir}')

## DestinE — load historical and future pr

In [ ]:
# DestinE historical (simh)
de_hist = DestinEHistoricalForcing.load(settings['path_DestinE_historical'])
da_de_hist_pr = xr.open_dataset(de_hist['pr'])['pr']
da_de_hist_pr['time'] = da_de_hist_pr['time'] + pd.Timedelta(hours=12)
da_de_hist_pr = da_de_hist_pr.sel(time=slice(cal_start, cal_end))
print(f'DestinE historical pr: {len(da_de_hist_pr.time)} timesteps')

In [ ]:
# DestinE future (simp)
de_future = DestinEFutureForcing.load(settings['path_DestinE'])
da_de_future_pr = xr.open_dataset(de_future['pr'])['pr']
da_de_future_pr['time'] = da_de_future_pr['time'] + pd.Timedelta(hours=12)
print(f'DestinE future pr: {len(da_de_future_pr.time)} timesteps')

# Read start/end dates from DestinE forcing YAML
de_yaml_path = Path(settings['path_DestinE']) / 'ewatercycle_forcing.yaml'
with open(de_yaml_path) as f:
    de_yaml = yaml.safe_load(f)
destine_start = de_yaml['start_time']
destine_end   = de_yaml['end_time']

## Apply bias correction — DestinE

In [ ]:
print('Bias correcting DestinE future pr ...')
da_de_bc = bc_pr(da_era5_pr, da_de_hist_pr, da_de_future_pr)

destine_bc_dir = Path(settings['path_DestinE']) / 'leakybucket_bias_correction'
save_lb_forcing(
    da_de_bc, destine_bc_dir,
    destine_start,
    destine_end,
    shape_path,
)
print(f'Saved → {destine_bc_dir}')

## Diagnostic plots — raw vs bias-corrected pr distributions

In [ ]:
def plot_pr_cdf(da_raw, da_bc, da_obs=None, title='', output_dir=None, name=None):
    """CDF comparison: raw, bias-corrected, and optionally ERA5."""
    fig, ax = plt.subplots(figsize=(7, 4))

    for da, label, color, ls in [
        (da_raw, 'Raw',            'steelblue', '--'),
        (da_bc,  'Bias-corrected', 'tomato',    '-'),
    ]:
        vals = np.sort(da.values.ravel())
        vals = vals[~np.isnan(vals)]
        cdf  = np.arange(1, len(vals) + 1) / len(vals)
        ax.plot(vals, cdf, color=color, ls=ls, lw=1.8, label=label)

    if da_obs is not None:
        vals = np.sort(da_obs.values.ravel())
        vals = vals[~np.isnan(vals)]
        cdf  = np.arange(1, len(vals) + 1) / len(vals)
        ax.plot(vals, cdf, color='black', lw=1.2, alpha=0.7, label='ERA5 (ref, cal. period)')

    ax.set_xlabel('pr (kg m⁻² s⁻¹)')
    ax.set_ylabel('Cumulative probability')
    ax.set_title(title)
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()

    if output_dir:
        Path(output_dir).mkdir(parents=True, exist_ok=True)
        fname = f'{name}_cdf.png' if name else 'lb_bc_cdf.png'
        fig.savefig(Path(output_dir) / fname, dpi=150, bbox_inches='tight')

    display(fig)
    plt.close(fig)

In [ ]:
figures_dir = Path(settings['figure_output']) / 'lb_bias_correction'
figures_dir.mkdir(parents=True, exist_ok=True)

# CMIP6 — plot first ensemble per scenario
for (dataset, experiment, ensemble), da_simp in cmip_future_pr.items():
    if ensemble != settings['CMIP_info']['ensembles'][0]:
        continue  # only plot one ensemble per scenario to keep output manageable

    da_bc = xr.open_dataset(
        cmip_bc_dirs[(dataset, experiment, ensemble)]
        / list(yaml.safe_load(
            open(cmip_bc_dirs[(dataset, experiment, ensemble)] / 'ewatercycle_forcing.yaml')
        )['filenames'].values())[0]
    )['pr']

    plot_pr_cdf(
        da_raw=da_simp,
        da_bc=da_bc,
        da_obs=da_era5_pr,
        title=f'LeakyBucket BC — {dataset}/{experiment}/{ensemble}',
        output_dir=figures_dir,
        name=f'CMIP6_{dataset}_{experiment}_{ensemble}',
    )

In [ ]:
# DestinE diagnostic
da_de_bc_reload = xr.open_dataset(
    destine_bc_dir / list(yaml.safe_load(
        open(destine_bc_dir / 'ewatercycle_forcing.yaml')
    )['filenames'].values())[0]
)['pr']

plot_pr_cdf(
    da_raw=da_de_future_pr,
    da_bc=da_de_bc_reload,
    da_obs=da_era5_pr,
    title='LeakyBucket BC — DestinE SSP3-7.0',
    output_dir=figures_dir,
    name='DestinE_SSP3-7.0',
)

## Verify — reload as LumpedMakkinkForcing

In [ ]:
# Verify one CMIP6 entry
first_key = next(iter(cmip_bc_dirs))
cmip_check = ewatercycle.forcing.sources['LumpedMakkinkForcing'].load(
    directory=cmip_bc_dirs[first_key]
)
display(cmip_check)

# Verify DestinE
destine_check = ewatercycle.forcing.sources['LumpedMakkinkForcing'].load(
    directory=destine_bc_dir
)
display(destine_check)

print('[bold green]All LeakyBucket bias-corrected forcing verified — ready for step 3.[/bold green]')